# 05 — Reinforcement Learning: PPO Training
**Goal**: Train the SFT model using Proximal Policy Optimization (PPO) with execution-guided sandbox rewards and KL divergence penalties (KL $\beta=0.02$, LR $1\times 10^{-6}$). Produces `./checkpoints/ppo/final`.

---

## Step 1: Environment & Universal Path Resolution

In [ ]:
import sys, os, shutil

# Universal Path Resolution & Auto-Copy for Kaggle / Local / Colab
def prepare_kaggle_src():
    curr = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(curr, 'src', 'models', 'loader.py')):
        print(f"Using local 'src' directory at {curr}")
        return curr
    
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'models' in dirs and os.path.exists(os.path.join(root, 'models', 'loader.py')):
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(root, dest)
                print(f"Copied 'src' from {root} to {dest}")
                return '/kaggle/working'
            elif 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'models', 'loader.py')):
                src_dir = os.path.join(root, 'src')
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(src_dir, dest)
                print(f"Copied 'src' from {src_dir} to {dest}")
                return '/kaggle/working'
    
    parent = os.path.abspath('..')
    if os.path.exists(os.path.join(parent, 'src')):
        return parent
    return curr

repo_root = prepare_kaggle_src()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from src.training.ppo import run_ppo_training
from src.utils.checkpoint import auto_checkpoint, guard_session_limit

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")

## Step 2: Load APPS Dataset & Tokenizer

In [ ]:
!pip uninstall -y torchao

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
SFT_CHECKPOINT = "./checkpoints/sft/final"

# Resolve SFT Checkpoint path on Kaggle input / working
if not os.path.exists(SFT_CHECKPOINT) and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files or 'model.safetensors' in files:
            SFT_CHECKPOINT = root
            break

effective_model = SFT_CHECKPOINT if os.path.exists(SFT_CHECKPOINT) else MODEL_NAME
print(f"Using SFT model checkpoint: {effective_model}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading APPS training dataset for PPO via Parquet branch...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:1000]')
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)
print(f"Prepared {len(apps_clean)} APPS problems for PPO rollout.")

## Step 3: Run PPO Training with Auto-Checkpointing

In [ ]:
session_start = time.time()
print("Starting PPO Reinforcement Learning Training loop...")

ppo_trainer = run_ppo_training(
    sft_model_path=effective_model,
    tokenizer=tokenizer,
    dataset=apps_clean,
    output_dir="./checkpoints/ppo",
    num_epochs=1,
    learning_rate=1e-6,
    batch_size=16,
    mini_batch_size=4,
    gradient_accumulation_steps=4,
    init_kl_coef=0.02,
    target_kl=6.0,
)

print("\nPPO Training completed successfully!")
print("Saved final PPO adapter checkpoint to ./checkpoints/ppo/final")